In [1]:
import json
from pathlib import Path

ordner = Path.home() / "Desktop" / "Modul 5" / "Aufgaben" / "Woche_3"
datei = ordner / "bestellungen_seite_1.json"

with open(datei, "r", encoding="utf-8") as f:
    daten = json.load(f)

print("Datentyp:", type(daten))
print("Inhalt:", daten)

Datentyp: <class 'dict'>
Inhalt: {'meta': {'seite': 1, 'pro_seite': 25, 'gesamt_bestellungen': 111, 'naechste_seite': 2}, 'daten': [{'bestell_id': 4001, 'datum': '2026-04-01', 'kunde': {'email': 'olivia.lange@example.com', 'land': 'DE'}, 'produkt': 'Maus', 'menge': 1, 'preis_eur': 19.9}, {'bestell_id': 4002, 'datum': '2026-04-01', 'kunde': {'email': 'lukas.schwarz@example.com', 'land': 'DE'}, 'produkt': 'USB-C-Kabel', 'menge': 1, 'preis_eur': 9.95}, {'bestell_id': 4003, 'datum': '2026-04-01', 'kunde': {'email': 'tim.klein@example.com', 'land': 'DE'}, 'produkt': 'Monitor 24 Zoll', 'menge': 3, 'preis_eur': 129.0}, {'bestell_id': 4004, 'datum': '2026-04-02', 'kunde': {'email': 'olivia.richter@example.com', 'land': 'DE'}, 'produkt': 'Dockingstation', 'menge': 4, 'preis_eur': 149.0}, {'bestell_id': 4005, 'datum': '2026-04-02', 'kunde': {'email': 'lukas.schwarz@example.com', 'land': 'AT'}, 'produkt': 'USB-C-Kabel', 'menge': 2, 'preis_eur': 9.95}, {'bestell_id': 4006, 'datum': '2026-04-03', '

In [2]:
print("Metadaten:", daten["meta"])
print("Anzahl Bestellungen:", len(daten["daten"]))
print("Erste Bestellung:", daten["daten"][0])

Metadaten: {'seite': 1, 'pro_seite': 25, 'gesamt_bestellungen': 111, 'naechste_seite': 2}
Anzahl Bestellungen: 25
Erste Bestellung: {'bestell_id': 4001, 'datum': '2026-04-01', 'kunde': {'email': 'olivia.lange@example.com', 'land': 'DE'}, 'produkt': 'Maus', 'menge': 1, 'preis_eur': 19.9}


In [3]:
for seite in range(1, 6):
    datei = ordner / f"bestellungen_seite_{seite}.json"

    with open(datei, "r", encoding="utf-8") as f:
        inhalt = json.load(f)

    print(f"Seite {seite}")
    print("Metadaten:", inhalt.get("meta"))
    print("Bestellungen:", len(inhalt.get("daten", [])))
    print("---")

Seite 1
Metadaten: {'seite': 1, 'pro_seite': 25, 'gesamt_bestellungen': 111, 'naechste_seite': 2}
Bestellungen: 25
---
Seite 2
Metadaten: {'seite': 2, 'pro_seite': 25, 'gesamt_bestellungen': 111, 'naechste_seite': 3}
Bestellungen: 25
---
Seite 3
Metadaten: {'seite': 3, 'pro_seite': 25, 'gesamt_bestellungen': 111, 'naechste_seite': 4}
Bestellungen: 25
---
Seite 4
Metadaten: {'seite': 4, 'pro_seite': 25, 'gesamt_bestellungen': 111, 'naechste_seite': 5}
Bestellungen: 25
---
Seite 5
Metadaten: {'seite': 5, 'pro_seite': 25, 'gesamt_bestellungen': 111, 'naechste_seite': None}
Bestellungen: 12
---


In [4]:
alle_bestellungen = []
seite = 1

while seite is not None:
    datei = ordner / f"bestellungen_seite_{seite}.json"

    with open(datei, "r", encoding="utf-8") as f:
        antwort = json.load(f)

    alle_bestellungen.extend(antwort["daten"])

    print(f"Seite {seite} eingelesen.")

    seite = antwort["meta"]["naechste_seite"]

print("Gesammelte Datensätze:", len(alle_bestellungen))

Seite 1 eingelesen.
Seite 2 eingelesen.
Seite 3 eingelesen.
Seite 4 eingelesen.
Seite 5 eingelesen.
Gesammelte Datensätze: 112


In [5]:
import pandas as pd

df = pd.json_normalize(alle_bestellungen)

print("Spalten:", df.columns.tolist())
print("Zeilen:", len(df))

dubletten = df[df.duplicated(subset=["bestell_id"], keep=False)]

display(dubletten)

Spalten: ['bestell_id', 'datum', 'produkt', 'menge', 'preis_eur', 'kunde.email', 'kunde.land']
Zeilen: 112


,bestell_id,datum,produkt,menge,preis_eur,kunde.email,kunde.land
74,4075,2026-04-24,Tastatur,3,39.9,katrin.becker@example.com,DE
75,4075,2026-04-24,Tastatur,3,39.9,katrin.becker@example.com,DE


In [6]:
print("Vollständig identische Dubletten:", df.duplicated().sum())

df_eindeutig = df.drop_duplicates().copy()

print("Vorher:", len(df))
print("Nachher:", len(df_eindeutig))
print("Entfernt:", len(df) - len(df_eindeutig))

assert len(df_eindeutig) == 111
assert df_eindeutig["bestell_id"].is_unique

Vollständig identische Dubletten: 1
Vorher: 112
Nachher: 111
Entfernt: 1


In [7]:
erwartet = daten["meta"]["gesamt_bestellungen"]
tatsaechlich = len(df_eindeutig)

fehlende_emails = df_eindeutig["kunde.email"].isna().sum()

print("Erwartete Bestellungen:", erwartet)
print("Tatsächliche Bestellungen:", tatsaechlich)
print("Fehlende E-Mail-Adressen:", fehlende_emails)

assert erwartet == tatsaechlich

print("Vollständigkeitsprüfung bestanden!")

Erwartete Bestellungen: 111
Tatsächliche Bestellungen: 111
Fehlende E-Mail-Adressen: 4
Vollständigkeitsprüfung bestanden!


In [8]:
artefakt_ordner = ordner / "artefakte"
artefakt_ordner.mkdir(exist_ok=True)

ziel = artefakt_ordner / "api_bestellungen_roh.parquet"

df_eindeutig.to_parquet(ziel, index=False)

kontrolle = pd.read_parquet(ziel)

assert len(kontrolle) == 111

print("Parquet erfolgreich gespeichert!")
print("Datei:", ziel)
print("Gespeicherte Bestellungen:", len(kontrolle))

Parquet erfolgreich gespeichert!
Datei: /Users/saschagolparvari/Desktop/Modul 5/Aufgaben/Woche_3/artefakte/api_bestellungen_roh.parquet
Gespeicherte Bestellungen: 111


In [9]:
from datetime import datetime, timezone

protokoll = {
    "zeitpunkt_utc": datetime.now(timezone.utc).isoformat(),
    "quelle": "Mock-API (lokale JSON-Dateien)",
    "eingelesene_seiten": 5,
    "datensaetze_roh": len(df),
    "transport_dubletten": len(df) - len(df_eindeutig),
    "entfernte_bestell_id": 4075,
    "datensaetze_eindeutig": len(df_eindeutig),
    "erwartete_bestellungen": erwartet,
    "fehlende_emails": int(fehlende_emails),
    "status": "erfolgreich"
}

protokoll_datei = artefakt_ordner / "api_ingest_protokoll.json"

with open(protokoll_datei, "w", encoding="utf-8") as f:
    json.dump(protokoll, f, indent=4, ensure_ascii=False)

print("Ingest-Protokoll gespeichert!")
print("Datei:", protokoll_datei)
print(json.dumps(protokoll, indent=4, ensure_ascii=False))

Ingest-Protokoll gespeichert!
Datei: /Users/saschagolparvari/Desktop/Modul 5/Aufgaben/Woche_3/artefakte/api_ingest_protokoll.json
{
    "zeitpunkt_utc": "2026-11-16T09:44:26.043250+00:00",
    "quelle": "Mock-API (lokale JSON-Dateien)",
    "eingelesene_seiten": 5,
    "datensaetze_roh": 112,
    "transport_dubletten": 1,
    "entfernte_bestell_id": 4075,
    "datensaetze_eindeutig": 111,
    "erwartete_bestellungen": 111,
    "fehlende_emails": 4,
    "status": "erfolgreich"
}


In [10]:
aufrufe = {}

def mock_api_abrufen(seite):
    aufrufe[seite] = aufrufe.get(seite, 0) + 1

    if seite == 3 and aufrufe[seite] == 1:
        return {"status_code": 429, "retry_after": 1}

    datei = ordner / f"bestellungen_seite_{seite}.json"

    with open(datei, "r", encoding="utf-8") as f:
        inhalt = json.load(f)

    return {"status_code": 200, "inhalt": inhalt}

print(mock_api_abrufen(3))

{'status_code': 429, 'retry_after': 1}


In [11]:
import time

def abrufen_mit_retry(seite, max_versuche=3):
    for versuch in range(1, max_versuche + 1):
        antwort = mock_api_abrufen(seite)
        status = antwort["status_code"]

        if status == 200:
            print(f"Seite {seite}: erfolgreich (Versuch {versuch})")
            return antwort["inhalt"]

        if status == 429:
            if versuch == max_versuche:
                raise RuntimeError("Maximale Anzahl an Versuchen erreicht")

            wartezeit = antwort["retry_after"]
            print(f"HTTP 429 – warte {wartezeit} Sekunde(n)")
            time.sleep(wartezeit)
            continue

        raise RuntimeError(f"Unerwarteter HTTP-Status: {status}")

    raise RuntimeError("Abruf fehlgeschlagen")


# Simulation zurücksetzen, damit Seite 3 erneut HTTP 429 liefert
aufrufe.clear()

ergebnis = abrufen_mit_retry(3)

print("Bestellungen auf Seite 3:", len(ergebnis["daten"]))

HTTP 429 – warte 1 Sekunde(n)
Seite 3: erfolgreich (Versuch 2)
Bestellungen auf Seite 3: 25


In [12]:
aufrufe.clear()

alle_bestellungen_retry = []
seite = 1

while seite is not None:
    antwort = abrufen_mit_retry(seite)

    alle_bestellungen_retry.extend(antwort["daten"])
    seite = antwort["meta"]["naechste_seite"]

print("Gesammelte Datensätze:", len(alle_bestellungen_retry))
print("Aufrufe pro Seite:", aufrufe)

assert len(alle_bestellungen_retry) == 112
assert aufrufe[3] == 2

print("API-Abruf inklusive Retry erfolgreich!")

Seite 1: erfolgreich (Versuch 1)
Seite 2: erfolgreich (Versuch 1)
HTTP 429 – warte 1 Sekunde(n)
Seite 3: erfolgreich (Versuch 2)
Seite 4: erfolgreich (Versuch 1)
Seite 5: erfolgreich (Versuch 1)
Gesammelte Datensätze: 112
Aufrufe pro Seite: {1: 1, 2: 1, 3: 2, 4: 1, 5: 1}
API-Abruf inklusive Retry erfolgreich!


### KI-Reflexion – Woche 3, Tag 2

Für diese Aufgabe habe ich ChatGPT als Lern- und Programmierhilfe eingesetzt. GitHub Copilot habe ich nicht verwendet.

Mit ChatGPT habe ich die Verarbeitung der JSON-Dateien schrittweise erarbeitet. Den Python-Code habe ich selbst in Jupyter ausgeführt und die Ergebnisse überprüft.

Dabei habe ich gelernt, wie eine Seitennavigation über Metadaten funktioniert, wie sich JSON-Daten mit `pd.json_normalize()` in eine Tabelle umwandeln lassen und wie man Transport-Dubletten erkennt.

Außerdem habe ich eine HTTP-429-Fehlerbehandlung mit Wartezeit und erneutem Versuch getestet. Da die Daten lokal vorlagen, wurde die HTTP-Antwort simuliert.

Die vier fehlenden E-Mail-Adressen habe ich bewusst nicht verändert, sondern dokumentiert.

**Mein Fazit:** KI kann beim Programmieren und Lernen helfen. Wichtig ist aber, den Code selbst nachzuvollziehen, auszuführen und die Ergebnisse zu kontrollieren.